In [113]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA


In [114]:
df=pd.read_csv("flats_missing_val.csv")

In [115]:
df.shape

(3034, 26)

In [116]:
drop_cols = [
    'link',
    'society',
    'area',
    'price_per_sqft',
    'lounge_or_sitting_room','other_rooms','dining_room','steam_room',
 'gym','powder_room','prayer_room','laundry_room','study_room'
]

df=df.drop(columns=drop_cols)

In [117]:
df.columns

Index(['bedrooms', 'baths', 'floors_in_building', 'price', 'area_sqft',
       'servant_quarters', 'kitchens', 'store_rooms', 'drawing_room',
       'luxury_score', 'furnishing_score', 'floor', 'agePossession'],
      dtype='object')

In [118]:
def categorize_luxury(score):
    if score < 120:
        return "Low"
    elif 120 <= score < 155:
        return "Medium"
    elif 155 <= score < 175:
        return "High"
    else:
        return "Very High"
    
    
q1 = df['luxury_score'].quantile(0.25)
q2 = df['luxury_score'].quantile(0.50)
q3 = df['luxury_score'].quantile(0.75)

def categorize_luxury(score):
    if score < q1:
        return "Low"
    elif score < q2:
        return "Medium"
    elif score < q3:
        return "High"
    else:
        return "Very High"


In [119]:
df['luxury_category'] = df['luxury_score'].apply(categorize_luxury)

In [120]:
def categorize_floor(floor):
    if 0 <= floor <= 2:
        return "Low Floor"
    elif 3 <= floor <= 10:
        return "Mid Floor"
    elif floor >= 11:
        return "High Floor"
    else:
        return None

df['floor_category'] = df['floor'].apply(categorize_floor)

In [121]:
df.drop(columns=['floor', 'luxury_score'], inplace=True)

In [122]:
df.head()

,bedrooms,baths,floors_in_building,price,area_sqft,servant_quarters,kitchens,store_rooms,drawing_room,furnishing_score,agePossession,luxury_category,floor_category
0,4.0,4.0,9.0,4.55,3264.0,1.0,1.0,1.0,1,37,New Property,Medium,Mid Floor
1,4.0,5.0,10.0,4.50,3318.4,1.0,1.0,0.0,1,31,New Property,Medium,Mid Floor
2,3.0,4.0,8.0,3.55,2720.0,1.0,1.0,1.0,1,49,New Property,Medium,Mid Floor
3,4.0,4.0,8.0,4.45,2067.2,1.0,2.0,1.0,1,18,New Property,Medium,Mid Floor
4,4.0,5.0,10.0,4.65,3318.4,1.0,1.0,1.0,1,18,New Property,Low,Mid Floor


In [123]:
df.shape

(3034, 13)

In [124]:
df['furnishing_score'].value_counts()

furnishing_score
49    1445
39     518
0      139
12     105
47      88
16      77
18      72
37      68
24      67
22      64
33      42
30      34
31      31
28      28
9       26
7       22
40      19
34      19
32      16
6       16
19      15
43      13
41      12
23      12
2       11
26      11
25       9
15       9
38       8
17       8
21       8
13       7
42       6
27       5
11       2
29       1
35       1
Name: count, dtype: int64

In [125]:
df['furnishing_type'] = pd.cut(
    df['furnishing_score'],
    bins=[-1, 0, 10, 25, 100],
    labels=[
        'Unfurnished',
        'Semi-Furnished',
        'Furnished',
        'Luxury Furnished'
    ]
)

In [126]:
df=df.drop(columns='furnishing_score')

In [127]:
df.head()

,bedrooms,baths,floors_in_building,price,area_sqft,servant_quarters,kitchens,store_rooms,drawing_room,agePossession,luxury_category,floor_category,furnishing_type
0,4.0,4.0,9.0,4.55,3264.0,1.0,1.0,1.0,1,New Property,Medium,Mid Floor,Luxury Furnished
1,4.0,5.0,10.0,4.50,3318.4,1.0,1.0,0.0,1,New Property,Medium,Mid Floor,Luxury Furnished
2,3.0,4.0,8.0,3.55,2720.0,1.0,1.0,1.0,1,New Property,Medium,Mid Floor,Luxury Furnished
3,4.0,4.0,8.0,4.45,2067.2,1.0,2.0,1.0,1,New Property,Medium,Mid Floor,Furnished
4,4.0,5.0,10.0,4.65,3318.4,1.0,1.0,1.0,1,New Property,Low,Mid Floor,Furnished


In [128]:
df.columns

Index(['bedrooms', 'baths', 'floors_in_building', 'price', 'area_sqft',
       'servant_quarters', 'kitchens', 'store_rooms', 'drawing_room',
       'agePossession', 'luxury_category', 'floor_category',
       'furnishing_type'],
      dtype='object')

In [129]:
df['bedrooms'].value_counts()

bedrooms
1.00000    1089
2.00000     815
3.00000     715
4.00000     361
0.00000      43
2.09154       8
6.00000       2
9.00000       1
Name: count, dtype: int64

In [130]:
df['baths'].value_counts()

baths
1.0    1155
2.0     732
3.0     539
4.0     304
5.0     296
6.0       7
7.0       1
Name: count, dtype: int64

In [131]:
df['servant_quarters'].value_counts()

servant_quarters
1.0    1487
0.0    1362
2.0     170
3.0      10
4.0       3
5.0       1
6.0       1
Name: count, dtype: int64

In [132]:
df['kitchens'].value_counts()

kitchens
1.0    2752
2.0     255
3.0      27
Name: count, dtype: int64

In [133]:
df['store_rooms'].value_counts()

store_rooms
1.0    1563
0.0    1326
2.0     130
3.0      13
4.0       2
Name: count, dtype: int64

In [134]:
df['drawing_room'] = df['drawing_room'].map({1: 'Yes', 0: 'No'})

In [135]:
df['drawing_room'].value_counts()

drawing_room
Yes    2405
No      629
Name: count, dtype: int64

In [136]:
# Split features and target
X = df.drop(columns=['price'])
y = df['price']

In [137]:
# Apply log1p transformation
y_transformed = np.log1p(y)

In [138]:
X.columns

Index(['bedrooms', 'baths', 'floors_in_building', 'area_sqft',
       'servant_quarters', 'kitchens', 'store_rooms', 'drawing_room',
       'agePossession', 'luxury_category', 'floor_category',
       'furnishing_type'],
      dtype='object')

In [139]:
df.columns

Index(['bedrooms', 'baths', 'floors_in_building', 'price', 'area_sqft',
       'servant_quarters', 'kitchens', 'store_rooms', 'drawing_room',
       'agePossession', 'luxury_category', 'floor_category',
       'furnishing_type'],
      dtype='object')

In [140]:
df.duplicated().sum()

np.int64(250)

In [141]:
X.dtypes

bedrooms               float64
baths                  float64
floors_in_building     float64
area_sqft              float64
servant_quarters       float64
kitchens               float64
store_rooms            float64
drawing_room            object
agePossession           object
luxury_category         object
floor_category          object
furnishing_type       category
dtype: object

### ordinal encoding

In [142]:
num_cols = [
    'bedrooms',
    'baths',
    'area_sqft',
    'kitchens',
    'store_rooms',
    'floors_in_building',
    'servant_quarters'
]

ordinal_cols = [
    'drawing_room',
    'agePossession',
    'luxury_category',
    'floor_category',
    'furnishing_type'
]



In [143]:
X.isnull().sum()

bedrooms              0
baths                 0
floors_in_building    0
area_sqft             0
servant_quarters      0
kitchens              0
store_rooms           0
drawing_room          0
agePossession         0
luxury_category       0
floor_category        0
furnishing_type       0
dtype: int64

In [144]:
X['floor_category'].value_counts()

floor_category
Mid Floor     2390
Low Floor      526
High Floor     118
Name: count, dtype: int64

In [145]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OrdinalEncoder(), ordinal_cols),
    ],
    remainder='passthrough'
)

In [146]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [147]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

scores.mean(), scores.std()

(np.float64(0.6986082748003081), np.float64(0.03448610708170978))

In [148]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)

In [149]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedrooms', 'baths',
                                                   'area_sqft', 'kitchens',
                                                   'store_rooms',
                                                   'floors_in_building',
                                                   'servant_quarters']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['drawing_room',
                                                   'agePossession',
                                                   'luxury_category',
                                                   'floor_category',
                                                   'furnishing_type'])])),
                ('regressor', LinearRegression())])

In [150]:
y_pred = pipeline.predict(X_test)

In [151]:
y_pred = np.expm1(y_pred)

In [152]:
mean_absolute_error(np.expm1(y_test), y_pred)

0.6251562411060578

In [153]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_transformed, test_size=0.2, random_state=42
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test), y_pred))

    return output

In [154]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

model_dict = {
    'linear_reg': LinearRegression(),
    'svr': SVR(),
    'ridge': Ridge(),
    'lasso': Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest': RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost': XGBRegressor()
}

In [100]:
model_output = []
for model_name, model in model_dict.items():
    model_output.append(scorer(model_name, model))

KeyboardInterrupt: 

In [ ]:
model_output

[['linear_reg', np.float64(0.6986082748003081), 0.6251562411060578],
 ['svr', np.float64(0.746991767999431), 0.539441657528102],
 ['ridge', np.float64(0.6986144805683834), 0.6251316151949863],
 ['lasso', np.float64(-0.006863961950449094), 1.4006642187155462],
 ['decision tree', np.float64(0.6937780056390987), 0.6328085592837819],
 ['random forest', np.float64(0.8147415899817888), 0.4830968587415066],
 ['extra trees', np.float64(0.7986630872310513), 0.4893235529408343],
 ['gradient boosting', np.float64(0.8118822599422018), 0.4900150070026386],
 ['adaboost', np.float64(0.7170865463137152), 0.6777667281474044],
 ['mlp', np.float64(0.737441204596201), 0.5519759950138777],
 ['xgboost', np.float64(0.8100226386979001), 0.5014682399380148]]

In [ ]:
model_df = pd.DataFrame(model_output, columns=['name', 'r2', 'mae'])

In [ ]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.814742,0.483097
6,extra trees,0.798663,0.489324
7,gradient boosting,0.811882,0.490015
10,xgboost,0.810023,0.501468
1,svr,0.746992,0.539442
9,mlp,0.737441,0.551976
2,ridge,0.698614,0.625132
0,linear_reg,0.698608,0.625156
4,decision tree,0.693778,0.632809
8,adaboost,0.717087,0.677767


## one hoe encoding

In [155]:
df.columns

Index(['bedrooms', 'baths', 'floors_in_building', 'price', 'area_sqft',
       'servant_quarters', 'kitchens', 'store_rooms', 'drawing_room',
       'agePossession', 'luxury_category', 'floor_category',
       'furnishing_type'],
      dtype='object')

In [156]:
preprocessor = ColumnTransformer(
    transformers=[
        
        # Numerical columns
        ('num', StandardScaler(), [
            'bedrooms',
            'baths',
            'area_sqft',
            'kitchens',
            'store_rooms',
            'servant_quarters',
            'floors_in_building'
        ]),
        
        # Ordinal columns (IMPORTANT: ordered)
        ('ord', OrdinalEncoder(), [
            'luxury_category',
            'floor_category',
            'furnishing_type'
        ]),
        
        # Binary / nominal (no order)
        ('cat1', OneHotEncoder(drop='first'), [
            'drawing_room','agePossession'
        ])
    ],
    
    remainder='passthrough'
)

In [157]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [158]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

scores.mean(), scores.std()

(np.float64(0.7021104211372444), np.float64(0.03555721640898394))

In [159]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)

In [160]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedrooms', 'baths',
                                                   'area_sqft', 'kitchens',
                                                   'store_rooms',
                                                   'servant_quarters',
                                                   'floors_in_building']),
                                                 ('ord', OrdinalEncoder(),
                                                  ['luxury_category',
                                                   'floor_category',
                                                   'furnishing_type']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first'),
                                                  ['drawing_room',
                                                   'agePossession'])])),
                ('regressor', LinearRegression())])

In [161]:
y_pred = pipeline.predict(X_test)

In [162]:
y_pred = np.expm1(y_pred)

In [163]:
mean_absolute_error(np.expm1(y_test), y_pred)

0.6215311387521316

In [ ]:
model_output = []
for model_name, model in model_dict.items():
    model_output.append(scorer(model_name, model))

c:\Users\zeeshan_ahmed\miniconda3\envs\ml_env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
model_df = pd.DataFrame(model_output, columns=['name', 'r2', 'mae'])
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.799163,0.484715
5,random forest,0.812077,0.484827
7,gradient boosting,0.811832,0.490868
10,xgboost,0.810023,0.501468
1,svr,0.746992,0.539442
9,mlp,0.739810,0.558392
2,ridge,0.698614,0.625132
0,linear_reg,0.698608,0.625156
4,decision tree,0.695947,0.626792
8,adaboost,0.713618,0.629091


In [ ]:
X.columns

Index(['bedrooms', 'baths', 'floors_in_building', 'area_sqft',
       'servant_quarters', 'kitchens', 'store_rooms', 'drawing_room',
       'agePossession', 'luxury_category', 'floor_category',
       'furnishing_type'],
      dtype='object')

In [164]:
X['drawing_room'] = X['drawing_room'].map({'Yes': 1, 'No': 0})

In [165]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# Numerical
num_cols = [
    'bedrooms',
    'baths',
    'floors_in_building',
    'area_sqft',
    'kitchens',
    'store_rooms',
    'servant_quarters'
]

# Ordered categorical
ordinal_cols = [
    'agePossession',
    'luxury_category',
    'floor_category',
    'furnishing_type'
]

# Binary (must be 0/1)
binary_cols = [
    'drawing_room'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('ord', OrdinalEncoder(), ordinal_cols),
        ('bin', 'passthrough', binary_cols)
    ],
    remainder='drop'
)

In [166]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [167]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

scores.mean(), scores.std()

(np.float64(0.6986082748003081), np.float64(0.03448610708170973))

In [168]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_transformed, test_size=0.2, random_state=42
)

In [169]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['bedrooms', 'baths',
                                                   'floors_in_building',
                                                   'area_sqft', 'kitchens',
                                                   'store_rooms',
                                                   'servant_quarters']),
                                                 ('ord', OrdinalEncoder(),
                                                  ['agePossession',
                                                   'luxury_category',
                                                   'floor_category',
                                                   'furnishing_type']),
                                                 ('bin', 'passthrough',
                                                  ['drawing_room'])])),
                ('regressor', LinearRegression())])

In [170]:
y_pred = pipeline.predict(X_test)

In [171]:
y_pred = np.expm1(y_pred)
mean_absolute_error(np.expm1(y_test), y_pred)

0.6251562411060578

In [ ]:
model_output = []
for model_name, model in model_dict.items():
    model_output.append(scorer(model_name, model))

KeyboardInterrupt: 

In [ ]:
model_df = pd.DataFrame(model_output, columns=['name', 'r2', 'mae'])
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.810994,0.476763
6,extra trees,0.799669,0.480484
7,gradient boosting,0.811842,0.490015
10,xgboost,0.809877,0.503492
1,svr,0.746992,0.539442
9,mlp,0.739028,0.544245
8,adaboost,0.719728,0.613837
2,ridge,0.698614,0.625132
0,linear_reg,0.698608,0.625156
4,decision tree,0.699089,0.626431


In [172]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__max_samples': [0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': ['auto', 'sqrt']
}

In [173]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('ord', OrdinalEncoder(), ordinal_cols),
        ('bin', 'passthrough', binary_cols)
    ],
    remainder='drop'
)

In [174]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

In [175]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

search = GridSearchCV(
    pipeline,
    param_grid,
    cv=kfold,
    scoring='r2',
    n_jobs=-1,
    verbose=4
)

search.fit(X, y_transformed)

Fitting 10 folds for each of 128 candidates, totalling 1280 fits


c:\Users\zeeshan_ahmed\miniconda3\envs\ml_env\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
640 fits failed out of a total of 1280.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
489 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\zeeshan_ahmed\miniconda3\envs\ml_env\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\zeeshan_ahmed\miniconda3\envs\ml_env\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\zeeshan_ahmed\miniconda3\envs\ml_en

GridSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['bedrooms',
                                                                          'baths',
                                                                          'floors_in_building',
                                                                          'area_sqft',
                                                                          'kitchens',
                                                                          'store_rooms',
                                                                          'servant_quarters']),
                                                                        ('ord',
                                                                         OrdinalEncoder(),
                                                                         ['agePossession',
                                                                          'luxury_category',
                                                                          'floor_category',
                                                                          'furnishing_type']),
                                                                        ('bin',
                                                                         'passthrough',
                                                                         ['drawing_room'])])),
                                       ('regressor', RandomForestRegressor())]),
             n_jobs=-1,
             param_grid={'regressor__max_depth': [None, 10, 20, 30],
                         'regressor__max_features': ['auto', 'sqrt'],
                         'regressor__max_samples': [0.1, 0.25, 0.5, 1.0],
                         'regressor__n_estimators': [50, 100, 200, 300]},
             scoring='r2', verbose=4)

In [177]:
final_pipe = search.best_estimator_

In [178]:

search.best_params_


{'regressor__max_depth': 20,
 'regressor__max_features': 'sqrt',
 'regressor__max_samples': 1.0,
 'regressor__n_estimators': 200}

In [179]:

search.best_score_



np.float64(0.8117099867398737)

In [180]:
final_pipe.fit(X, y_transformed)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['bedrooms', 'baths',
                                                   'floors_in_building',
                                                   'area_sqft', 'kitchens',
                                                   'store_rooms',
                                                   'servant_quarters']),
                                                 ('ord', OrdinalEncoder(),
                                                  ['agePossession',
                                                   'luxury_category',
                                                   'floor_category',
                                                   'furnishing_type']),
                                                 ('bin', 'passthrough',
                                                  ['drawing_room'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=20, max_features='sqrt',
                                       max_samples=1.0, n_estimators=200))])

In [181]:
y_pred = final_pipe.predict(X_test)

In [182]:
y_pred = np.expm1(y_pred)

In [183]:
mean_absolute_error(np.expm1(y_test), y_pred)

0.22545023017091156

In [184]:
import joblib

joblib.dump(final_pipe, 'flats_final_pipeline.pkl')

['flats_final_pipeline.pkl']

In [ ]:
joblib.dump(final_pipe, 'flats_final_pipeline.pkl')

In [185]:
import pickle

with open('flats_df.pkl', 'wb') as file:
    pickle.dump(X, file)

In [186]:
X.columns

Index(['bedrooms', 'baths', 'floors_in_building', 'area_sqft',
       'servant_quarters', 'kitchens', 'store_rooms', 'drawing_room',
       'agePossession', 'luxury_category', 'floor_category',
       'furnishing_type'],
      dtype='object')

In [187]:
X.iloc[0]

bedrooms                           4.0
baths                              4.0
floors_in_building                 9.0
area_sqft                       3264.0
servant_quarters                   1.0
kitchens                           1.0
store_rooms                        1.0
drawing_room                         1
agePossession             New Property
luxury_category                 Medium
floor_category               Mid Floor
furnishing_type       Luxury Furnished
Name: 0, dtype: object

In [190]:
import pandas as pd
data = [[
    1.0,   # bedrooms
    1.0,   # baths
    0,   # floors_in_building
    598.951,  # area_sqft
    0,   # servant_quarters
    1.0,   # kitchens
    1.0,   # store_rooms
    1,     # drawing_room
    'New Property',
    'Medium',
    'Mid Floor',
    'Luxury Furnished'
]]

columns = [
    'bedrooms',
    'baths',
    'floors_in_building',
    'area_sqft',
    'servant_quarters',
    'kitchens',
    'store_rooms',
    'drawing_room',
    'agePossession',
    'luxury_category',
    'floor_category',
    'furnishing_type'
]

one_df = pd.DataFrame(data, columns=columns)

one_df

,bedrooms,baths,floors_in_building,area_sqft,servant_quarters,kitchens,store_rooms,drawing_room,agePossession,luxury_category,floor_category,furnishing_type
0,1.0,1.0,0,598.951,0,1.0,1.0,1,New Property,Medium,Mid Floor,Luxury Furnished


In [191]:
final_pipe.predict(one_df)

array([0.66239336])

In [ ]:
import joblib
import numpy as np

final_flats_pipeline = joblib.load('flats_final_pipeline.pkl')

EOFError: 

In [192]:
import os
os.path.getsize('flats_final_pipeline.pkl')

38367370

In [193]:
import os
os.path.getsize('house_final_pipeline.pkl')

56180458

In [176]:
import os
os.path.getsize('flats_df.pkl')

217070